# CAT-LM test generation (Colab, free T4 GPU)

Part of the CANDOR-D4J baselines (see `scripts/run_catlm.py`).

1. **Runtime → Change runtime type → T4 GPU.**
2. Run all cells. The upload cell asks for `prompts.jsonl` (from `results/catlm/prompts.jsonl`).
3. At the end `generations.jsonl` downloads. Put it at `results/catlm/generations.jsonl` and run
   `python3 scripts/run_catlm.py evaluate bugs/dev40.txt` on the Mac.

Settings = the CAT-LM paper / authors' `main_sample.py`: 10 samples per prompt, temperature 0.2,
top_k 50, top_p 0.95, max 512 new tokens, trailing `</s>` removed. Generation stops early once the
test method's braces close (only saves time; the Mac cuts each sample at the same point).
If the Colab session disconnects, just run all cells again: finished prompts are skipped.
No API keys or personal data are needed.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install "transformers==4.44.2" sentencepiece accelerate

In [ ]:
import os, json
from google.colab import files
if not os.path.exists('prompts.jsonl'):
    files.upload()          # choose prompts.jsonl
prompts = [json.loads(l) for l in open('prompts.jsonl')]
todo = [p for p in prompts if 'prompt_full' in p]
print(len(prompts), 'classes,', len(todo), 'with a prompt')

In [ ]:
import torch, time
from transformers import AutoModelForCausalLM, AutoTokenizer, StoppingCriteria, StoppingCriteriaList
tok = AutoTokenizer.from_pretrained('nikitharao/catlm', use_fast=False)
model = AutoModelForCausalLM.from_pretrained('nikitharao/catlm', torch_dtype=torch.float16).cuda().eval()
N_SAMPLES, MAX_NEW, WINDOW = 10, 512, 8192
KV_BYTES_PER_TOKEN = 2 * model.config.num_hidden_layers * model.config.hidden_size * 2   # K and V, fp16

In [ ]:
def method_closed(text):
    """True once 'public void test' + text has a complete method (same rule as run_catlm.first_method)."""
    text = "public void test" + text
    depth, opened, i, n = 0, False, 0, len(text)
    while i < n:
        c = text[i]
        if text.startswith("//", i):
            i = text.find("\n", i); i = n if i < 0 else i; continue
        if text.startswith("/*", i):
            j = text.find("*/", i + 2); i = n if j < 0 else j + 2; continue
        if c in "\"'":
            j = i + 1
            while j < n and text[j] != c:
                j += 2 if text[j] == "\\" else 1
            i = j + 1; continue
        if c == "{": depth, opened = depth + 1, True
        elif c == "}":
            depth -= 1
            if opened and depth == 0: return True
        i += 1
    return False

class MethodClosed(StoppingCriteria):
    def __init__(self, start): self.start = start
    def __call__(self, input_ids, scores, **kw):
        return torch.tensor([method_closed(tok.decode(row[self.start:], skip_special_tokens=True))
                             for row in input_ids], device=input_ids.device)

def encode(text):
    ids = tok(text, return_tensors="pt").input_ids
    if tok.decode(ids[0, -1]) == "</s>":          # the model was trained without </s>
        ids = ids[:, :-1]
    return ids

In [ ]:
done = set()
if os.path.exists('generations.jsonl'):
    done = {(g['bug'], g['class']) for g in map(json.loads, open('generations.jsonl'))}
limit = WINDOW - MAX_NEW
t_all = time.time()
for k, p in enumerate(todo):
    if (p['bug'], p['class']) in done:
        continue
    t = time.time()
    ids, used = encode(p['prompt_full']), 'full'
    if ids.shape[1] > limit:
        ids, used = encode(p['prompt_short']), 'short'
    if ids.shape[1] > limit:
        ids, used = ids[:, -limit:], 'short_truncated'
    ids = ids.cuda()
    L = ids.shape[1]
    free = torch.cuda.mem_get_info()[0] * 0.6
    per_call = max(1, min(N_SAMPLES, int(free // (KV_BYTES_PER_TOKEN * (L + MAX_NEW)))))
    samples = []
    torch.manual_seed(1000 + k)                    # fixed seed per prompt: repeatable
    while len(samples) < N_SAMPLES:
        n = min(per_call, N_SAMPLES - len(samples))
        with torch.no_grad():
            out = model.generate(ids, attention_mask=torch.ones_like(ids), do_sample=True,
                                 max_new_tokens=MAX_NEW, top_k=50, top_p=0.95, temperature=0.2,
                                 num_return_sequences=n, pad_token_id=tok.pad_token_id or 0,
                                 stopping_criteria=StoppingCriteriaList([MethodClosed(L)]))
        samples += [tok.decode(o[L:], skip_special_tokens=True) for o in out]
        torch.cuda.empty_cache()
    g = {'bug': p['bug'], 'class': p['class'], 'prompt_used': used, 'prompt_tokens': L,
         'samples': samples, 'gen_seconds': round(time.time() - t, 1)}
    with open('generations.jsonl', 'a') as f:
        f.write(json.dumps(g) + '\n')
    print(f"{k+1}/{len(todo)} {p['bug']} {p['class'].split('.')[-1]}: {used}, {L} tokens, "
          f"{g['gen_seconds']}s, closed {sum(method_closed(s) for s in samples)}/{N_SAMPLES}", flush=True)
print('all done in', round((time.time() - t_all) / 60, 1), 'min')

In [ ]:
files.download('generations.jsonl')